# TEST-104 OFFICIAL — chạm lần thứ nhất và duy nhất

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

## ⚠️ Đọc trước khi chạy bất cứ cell nào

Test-104 là held-out khoá kín, **chạm đúng một lần** (AGENTS.md §3.4, §10). Mọi lựa
chọn đã khoá bằng văn bản ở [`docs/TEST104_PREREGISTRATION.md`](../docs/TEST104_PREREGISTRATION.md),
commit **trước** khi notebook này chạy. Notebook cố ý **không có tham số nào để sửa**.

Sau khi chạy xong, **không được** đổi config, checkpoint, nhiệt độ `T`, hay ngưỡng
defer vì con số nhận được. Muốn có số test cho một cấu hình khác thì đó là **lần chạm
thứ hai**, phải xin phép lại và viết pre-registration mới.

## Đã khoá những gì

| khoản | giá trị |
|---|---|
| config | `baseline_3dpatch.yaml` + cache E4 (lesion_tight · 112×112×32 · per_phase) |
| bộ dự đoán chính | **ensemble 5 fold**, trung bình softmax |
| TTA | **không** — trên out-of-fold nó âm (−0.0150; bản mặt phẳng P=0.048) |
| E6b / EMA / pretrained | **không** — E6b null (P=0.92), E7/E8 chưa chạy |
| `T` | fit trên 394 ca out-of-fold, áp mù lên test. **Không fit trên test** |
| xếp hạng defer | `−epistemic` = bất đồng giữa 5 model, kèm dòng đối chứng `max-prob` |

## Vì sao ensemble 5 fold hợp lệ ở đây mà bị cấm trên out-of-fold

Không model nào trong 5 cái từng thấy 104 ca này: cả 5 chỉ train trên tập con của 394
ca trainval, và `Splits.validate()` khẳng định `val_fold_i ∩ test = ∅` với mọi `i`.
Trên out-of-fold thì ngược lại — mỗi ca ở val của fold `f` nằm trong tập train của cả
4 model kia (AGENTS.md §3, WORKLOG S-080).

Đây cũng là chỗ duy nhất có **deep ensemble thật**, nên bất định epistemic đo được ở
đây tốt hơn MC-dropout — thứ đang phải trả giá −0.10 macro-F1 để có.

## Notebook này KHÔNG in metric nào

Có chủ đích. Nó chỉ suy luận và lưu `test_probs.npz`. Toàn bộ bảng số chạy ở máy local
bằng `python -m src.eval.test_report`, vì phần đó cần thư mục out-of-fold để lấy `T`,
và vì in ra cả bảng một lượt thì không có chỗ cho việc nhìn một con số rồi cân nhắc
lại. Lý do đầy đủ ở docstring của `src/eval/test_once.py`.

## Cần mount hai dataset

1. **cache E4** — nhận diện bằng `cache_meta.json`, không bằng tên
2. **checkpoint** — `best_fold_1..5.pt`; sha256 bị ghim sẵn trong code, lệch là dừng

## 0. Bootstrap

Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- KHÔNG CÓ THAM SỐ NÀO ĐỂ SỬA -------------------------------------------
# Mọi lựa chọn đã khoá ở docs/TEST104_PREREGISTRATION.md. Sửa gì ở đây là phá
# pre-registration, và con số ra sẽ không dùng được để báo cáo.
FOLDS = [1, 2, 3, 4, 5]
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/test104"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
print(f"config: {CFG_PATH.name} (E4 = config này + cache per-phase)")

## 1. Cache E4 và checkpoint

Cần **hai** thứ mount vào: cache E4 (`lesion_tight · 112×112×32 · per_phase`) và 5 file
`best.pt`. Đổi đường dẫn bên dưới cho khớp tên dataset bạn đã upload.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

# Cache E4 nhận diện bằng NỘI DUNG `cache_meta.json`, không bằng tên dataset. Tên do
# người upload đặt và đã lệch một lần rồi (`lld-mmri-lesion-tight/cache_lesion_tight`
# chứ không phải `lld-mmri-e4-per-phase` như đoán ở S-080). Ba khoá này là thứ phân
# biệt E4 với mọi cache trước đó.
E4_KEYS = {
    "align_phases": "per_phase",          # <- phân biệt E4 với E3
    "target_size": [112, 112, 32],        # <- phân biệt E3/E4 với E0/E1
    "crop_mode": "lesion_tight",          # <- phân biệt E1+ với E0
}

# Tên file checkpoint. KHÔNG kèm thư mục cha — độ sâu do `rglob` lo, xem bên dưới.
#   A) best_fold_1.pt ... best_fold_5.pt   <- dataset "best weights"
#   B) fold_1/best.pt ...                   <- gói thẳng từ output run
CKPT_NAMES = ["best_fold_{f}.pt", "best.pt"]

# ---------------------------------------------------------------------------
# KHÔNG hardcode độ sâu. Kaggle mount ở `/kaggle/input/datasets/<user>/<slug>/...`
# chứ không phải `/kaggle/input/<slug>/...` như mọi notebook trước giả định, và độ
# sâu đó có thể đổi tiếp. Dò theo TÊN FILE mốc, sâu bao nhiêu cũng thấy. Đây là lần
# thứ tư sửa cùng một lớp lỗi (S-081 → S-084); nguyên nhân gốc luôn là một giả định
# về hình dạng đường dẫn.
#
# MỘT lượt `os.walk` duy nhất thu hết mọi thứ cần. Không dùng nhiều `rglob` riêng:
# dataset gốc là 83.7GB / ~4000 file trên ổ mạng, và mỗi `rglob` là một lượt duyệt
# toàn cây — 11 lượt thì chờ rất lâu mà chẳng được gì thêm.
# ---------------------------------------------------------------------------
import json as _json
import os as _os
import re as _re

_cfg_data = load_yaml(REPO / "configs" / "data.yaml")
_ann_name = Path(_cfg_data["annotation_rel"]).name

interesting = {}       # thư mục -> số .npz/.pt/meta, để in bảng chẩn đoán
meta_paths = []        # cache_meta.json tìm được
ckpt_paths = []        # mọi file .pt tên best*.pt
_ann = []              # file annotation của dữ liệu gốc

for dirpath, dirnames, filenames in _os.walk(INPUT_ROOT):
    dirnames[:] = [x for x in dirnames if x not in (".cache", ".git")]  # rác tải HF
    here = {"npz": 0, "pt": 0, "meta": 0}
    for name in filenames:
        full = Path(dirpath) / name
        if name.endswith(".npz"):
            here["npz"] += 1
        elif name.endswith(".pt"):
            here["pt"] += 1
            if name.startswith("best"):
                ckpt_paths.append(full)
        elif name == "cache_meta.json":
            here["meta"] += 1
            meta_paths.append(full)
        elif name == _ann_name:
            _ann.append(full)
    if any(here.values()):
        interesting[Path(dirpath)] = here


def read_caches(paths):
    out = []
    for p in sorted(paths):
        try:
            out.append((p.parent, _json.loads(p.read_text("utf-8"))))
        except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo, không nuốt lỗi thật
            out.append((p.parent, {"__loi__": repr(exc)}))
    return out


def matches_e4(meta):
    return all(meta.get(k) == v for k, v in E4_KEYS.items())


def pick_checkpoints(paths, folds):
    """{fold: đường dẫn}. Ưu tiên `best_fold_N.pt`; `best.pt` thì suy fold từ thư
    mục cha (`fold_3/best.pt`). Không suy được thì bỏ, không đoán bừa."""
    out = {}
    for fold in folds:
        hits = [p for p in sorted(paths) if p.name == f"best_fold_{fold}.pt"]
        if not hits:
            hits = [
                p for p in sorted(paths)
                if p.name == "best.pt"
                and (m := _re.search(r"fold_?(\d+)", p.parent.name))
                and int(m.group(1)) == fold
            ]
        if hits:
            out[fold] = hits[0]
    return out


print(f"=== Thư mục có dữ liệu dưới {INPUT_ROOT} ===")
for d in sorted(interesting)[:25]:
    c = interesting[d]
    print(f"  {d}\n      {' · '.join(f'{c[k]} {k}' for k in ('npz', 'pt', 'meta') if c[k])}")
if not interesting:
    print("  (trống — chưa mount dataset nào có .npz/.pt)")

print(f"\n=== Dữ liệu gốc ({_ann_name}) ===")
for p in sorted(_ann)[:5]:
    print(f"  ✓ {p.parent.parent}")
if not _ann:
    print("  KHÔNG thấy — chỉ cần nếu phải build cache (xem ngay dưới)")

caches = read_caches(meta_paths)
print(f"\n=== {len(caches)} cache có cache_meta.json ===")
for path, meta in caches:
    mark = "✓ E4" if matches_e4(meta) else "  --"
    print(
        f"  {mark}  {path}\n"
        f"        crop={meta.get('crop_mode')} size={meta.get('target_size')} "
        f"align={meta.get('align_phases')}"
    )

e4 = [p for p, m in caches if matches_e4(m)]
if e4:
    CACHE_DIR = e4[0]
    BUILD_NEEDED = False
else:
    # Thư mục có nhiều .npz nhưng KHÔNG có meta: không dùng được, và phải nói rõ vì sao.
    # Hình dạng mảng cho biết target_size, nhưng KHÔNG cho biết `align_phases` —
    # E3 (reference) và E4 (per_phase) có cùng shape [8,112,112,32]. Nhận nhầm E3
    # thành E4 sẽ cho ra một bảng kết quả sai mà trông hoàn toàn hợp lý.
    for d, c in sorted(interesting.items()):
        if c["npz"] > 100 and not c["meta"]:
            print(f"\n⚠ {d} có {c['npz']} file .npz nhưng KHÔNG có cache_meta.json.")
            print("  Không dùng được: shape cho biết target_size nhưng KHÔNG phân biệt được")
            print("  E3 (align=reference) với E4 (align=per_phase) — hai cái cùng shape.")
    BUILD_NEEDED = True
    CACHE_DIR = Path("/kaggle/working/cache_e4")
    if _ann:
        print("\n=> sẽ BUILD lại cache E4 (~26 phút). Dữ liệu gốc đã có ✓")
    else:
        print(
            "\n=> CẦN BUILD cache E4 nhưng CHƯA MOUNT dữ liệu gốc.\n"
            f"   Mount dataset chứa {_cfg_data['annotation_rel']} "
            f"(ứng viên: {_cfg_data.get('data_root_candidates')}),\n"
            "   rồi chạy lại từ cell này."
        )

CKPTS = pick_checkpoints(ckpt_paths, FOLDS)
thieu = [f for f in FOLDS if f not in CKPTS]
assert not thieu, (
    f"không thấy checkpoint cho fold {thieu}.\n"
    f"Đã dò theo tên {CKPT_NAMES} ở MỌI độ sâu dưới {INPUT_ROOT}.\n"
    f"Tìm được: { {f: str(p) for f, p in CKPTS.items()} }"
)
print("\ncache:      ", CACHE_DIR, "(CHƯA CÓ — sẽ build ở cell dưới)" if BUILD_NEEDED else "")

# 5 file cùng kiến trúc nên cùng kích thước — kích thước KHÔNG chứng minh chúng khác
# nhau. Băm để chắc không phải một file bị chép 5 lần với 5 cái tên.
import hashlib

print("checkpoint:")
digests = {}
for f in FOLDS:
    p = CKPTS[f]
    h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    digests[f] = h
    print(f"  fold {f}: {p.name}  {p.stat().st_size / 2**20:.1f} MB  sha256 {h}")
assert len(set(digests.values())) == len(FOLDS), f"có checkpoint trùng nhau: {digests}"

# Đối chiếu với mã băm đo ở máy local (WORKLOG S-081). Khác => file khác bản.
LOCAL_SHA = {
    1: "2e1f3e1ad477ad59", 2: "30a8eb9ee221d453", 3: "00c133e031bdf8fe",
    4: "3fe18f1eb3de4431", 5: "d61cc7ed94b8ebf0",
}
lech = {f: (digests[f], LOCAL_SHA[f]) for f in FOLDS if f in LOCAL_SHA and digests[f] != LOCAL_SHA[f]}
if lech:
    print(f"\n⚠ mã băm khác bản local: {lech}")
    print("  Không tự dừng — nhưng nếu bạn không cố ý đổi checkpoint thì hãy dừng lại xem.")

## 1b. Build cache E4 nếu chưa có

Chỉ chạy khi cell trên không tìm thấy cache E4 nào. Build lại **cho ra đúng cùng dữ
liệu** — pipeline tiền xử lý tất định, `set_seed` chỉ ảnh hưởng train.

`resolve_data_root` tự lùng dataset LLD-MMRI gốc dưới `/kaggle/input` bằng cách tìm
file annotation, nên không cần khai đường dẫn. Nếu chưa mount dataset gốc thì cell
này sẽ báo rõ chứ không build ra cache rỗng.

In [ ]:
if BUILD_NEEDED:
    from src.utils.io import resolve_data_root

    # Data root khai ở configs/data.yaml, KHÔNG ở preprocess_*.yaml — file preprocess
    # chỉ có tham số tiền xử lý. `build_cache` cũng đọc data.yaml (xem hàm main của
    # nó). Kiểm trước ở đây chỉ để fail nhanh, thay vì chết giữa job 26 phút.
    cfg_data = load_yaml(REPO / "configs" / "data.yaml")
    try:
        data_root = resolve_data_root(cfg_data)
    except Exception as exc:
        # RuntimeError chứ không SystemExit: SystemExit làm IPython lỗi khi dựng
        # traceback và che mất thông báo thật bằng một trang lỗi của chính nó.
        data_root, exc_msg = None, str(exc)
    else:
        exc_msg = None

    # `resolve_data_root` trả về `config['data_root']` mà KHÔNG xác minh khi mọi cách
    # dò đều trượt (src/utils/io.py:219-225). Trên Kaggle nó sẽ là `data/lldmmridataset`
    # tương đối, không tồn tại — và job 26 phút sẽ chết giữa chừng. Xác minh ở đây.
    ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
    if ann is None or not ann.exists():
        raise RuntimeError(
            f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
            f"  resolve_data_root -> {data_root}"
            + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
            + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
            f"  Ứng viên khai trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
        ) from None
    print("data root:", data_root, "✓")

    os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
    rc = subprocess.run(
        [sys.executable, "-m", "src.preprocess.build_cache",
         "--config", "configs/preprocess_e4.yaml"],
        cwd=REPO,
    ).returncode
    assert rc == 0, "build cache thất bại"
    print("build xong:", CACHE_DIR)
else:
    print("bỏ qua build — đã có cache E4:", CACHE_DIR)

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

## Cổng A ⚠️ — cache có đúng là E4 không

Chạy MC-dropout trên cache của E1 hay E3 sẽ **không báo lỗi gì cả**, chỉ lặng lẽ cho ra
số sai.

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))

# Dùng lại E4_KEYS của cell trên, không chép ra bản thứ hai — hai bản sẽ trôi khỏi nhau.
for key, want in E4_KEYS.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(Path(os.environ["LLDMMRI_CACHE_DIR"]).glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"
print(f"cache_meta khớp E4 ✓ · {n_npz} ca · commit {meta.get('git_commit')}")

## 2. Chạm test-104

Cell này là lần chạm. Nó tự kiểm bốn thứ trước khi chạy, và nổ chứ không cảnh báo:

1. `docs/TEST104_PREREGISTRATION.md` đã commit (kiểm bằng `git log`, không phải bằng
   sự tồn tại của file — file viết sau khi nhìn số thì vô nghĩa)
2. `Splits.validate()` — trong đó có `val_fold_i ∩ test = ∅` với mọi `i`
3. sha256 của 5 checkpoint khớp danh sách ghim
4. không có hai checkpoint trùng nhau (ensemble đếm một model hai lần thì con số vẫn
   trông hợp lý)

104 ca × 5 model nên chỉ mất khoảng một phút.

In [ ]:
import subprocess as _sp

from src.eval.test_once import run as touch_test

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

# Thư mục checkpoint: lấy cha chung của 5 file mà cell dò đã tìm được.
CKPT_DIR = Path(_os.path.commonpath([str(p.parent) for p in CKPTS.values()]))
print("checkpoint dir:", CKPT_DIR)
print("cache dir:     ", CACHE_DIR)
print("prereg commit: ", _sp.run(
    ["git", "log", "-1", "--format=%h %ad", "--date=short", "--",
     "docs/TEST104_PREREGISTRATION.md"],
    cwd=REPO, capture_output=True, text=True).stdout.strip() or "(CHƯA COMMIT — sẽ nổ)")

path = touch_test(
    ckpt_dir=CKPT_DIR,
    config_path=CFG_PATH,
    out_dir=os.environ["LLDMMRI_OUTPUT_DIR"],
    cache_dir=CACHE_DIR,
    folds=FOLDS,
)
print("\nđã lưu:", path)
print("\nKhông in metric ở đây — có chủ đích. Tải file về rồi chạy ở local:")
print("    python -m src.eval.test_report --run-dir runs/test104")

## 3. Gói mang về

`test_probs.npz` chỉ vài chục KiB (5 × 104 × 7 số thực). Giải nén **một lớp** vào
`runs/test104/` rồi chạy phần báo cáo ở local.

In [ ]:
import shutil

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
PACK = Path("/kaggle/working/test104_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

for name in ("test_probs.npz", "test_run_meta.json"):
    src = OUT_ROOT / name
    if src.exists():
        shutil.copy2(src, PACK / name)

for f in sorted(PACK.rglob("*")):
    print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**10:.1f} KiB")

print((PACK / "test_run_meta.json").read_text("utf-8"))

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz bản thân là zip; trình giải nén bung đệ quy
  sẽ biến nó thành thư mục và src.eval.* sẽ không thấy (đã dính hai lần, S-078).

Đặt vào runs/test104/ rồi chạy ở local:
    python -m src.eval.test_report --run-dir runs/test104
""")